# **Data Cleansing**

## Order Data CLeansing

In [1]:
#import project lip
from pyspark.sql.types import *
from delta.tables import DeltaTable

StatementMeta(, 2b2b6f3d-da55-457b-a1f7-ef908c1628f0, 3, Finished, Available, Finished, False)

In [4]:

last_load_date = spark.sql("""
    SELECT Last_Load_Date 
    FROM GamingPlatform_STG.dbo.ETL_Metadata
    WHERE Stage = 'STG_Gaming'
""").collect()[0]["Last_Load_Date"]

print(f" ODS_Gaming Last Load Date: {last_load_date}")

StatementMeta(, ed94887b-178f-4856-8dc0-adfd017effcf, 6, Finished, Available, Finished, False)

 ODS_Gaming Last Load Date: 1900-01-01 00:00:00


In [2]:
# last_load_date='1999-01-01 00:00:00'

StatementMeta(, 2b2b6f3d-da55-457b-a1f7-ef908c1628f0, 4, Finished, Available, Finished, False)

In [3]:
# load Data in dataframe
df_orders = spark.sql(f"""
SELECT *
FROM GamingPlatform_ODS.dbo.orders
WHERE ModifiedDate > '{last_load_date}'
""")

StatementMeta(, 2b2b6f3d-da55-457b-a1f7-ef908c1628f0, 5, Finished, Available, Finished, False)

In [4]:
#create View from DataFrame
df_orders.createOrReplaceTempView("vw_orders_raw")

StatementMeta(, 2b2b6f3d-da55-457b-a1f7-ef908c1628f0, 6, Finished, Available, Finished, False)

In [5]:
# 3. Clean Data
df_Orders_Cleaned = spark.sql("""
   SELECT DISTINCT

    CASE 
        WHEN TRIM(order_id) IS NULL OR TRIM(order_id) = '' 
            THEN 999999
        ELSE CAST(TRIM(order_id) AS INT)
    END AS order_id,

    CASE 
        WHEN TRIM(user_id) IS NULL OR TRIM(user_id) = '' 
            THEN 999999
        ELSE CAST(TRIM(user_id) AS INT)
    END AS user_id,

    CASE 
        WHEN TRIM(game_id) IS NULL OR TRIM(game_id) = '' 
            THEN 999999
        ELSE CAST(TRIM(game_id) AS INT)
    END AS game_id,

    CASE 
        WHEN TRIM(game_name) IS NULL OR TRIM(game_name) IN ('', 'N.A') 
            THEN 'N.A'
        ELSE LOWER(TRIM(game_name))
    END AS game_name,

    CASE 
        WHEN TRIM(genre) IS NULL OR TRIM(genre) IN ('', 'N.A') 
            THEN 'N.A'
        ELSE LOWER(TRIM(genre))
    END AS genre,

    CASE 
        WHEN TRIM(country) IS NULL OR TRIM(country) IN ('', 'N.A') 
            THEN 'N.A'
        ELSE LOWER(TRIM(country))
    END AS country,

    CASE 
        WHEN TRIM(order_date) IS NULL OR TRIM(order_date) = '' 
            THEN CAST('1900-01-01' AS TIMESTAMP)
        ELSE TO_TIMESTAMP(TRIM(order_date))
    END AS order_date,

    CASE 
        WHEN TRIM(price) IS NULL OR TRIM(price) = '' 
            THEN CAST(999.999 AS DECIMAL(10,2))
        ELSE CAST(TRIM(price) AS DECIMAL(10,2))
    END AS price,

    CASE 
        WHEN TRIM(quantity) IS NULL OR TRIM(quantity) = '' 
            THEN 999999
        ELSE CAST(TRIM(quantity) AS INT)
    END AS quantity,

    CASE 
        WHEN TRIM(discount_amount) IS NULL OR TRIM(discount_amount) = '' 
            THEN CAST(999.999 AS DECIMAL(10,2))
        ELSE CAST(TRIM(discount_amount) AS DECIMAL(10,2))
    END AS discount_amount,

    CASE 
        WHEN TRIM(tax_amount) IS NULL OR TRIM(tax_amount) = '' 
            THEN CAST(999.999 AS DECIMAL(10,2))
        ELSE CAST(TRIM(tax_amount) AS DECIMAL(10,2))
    END AS tax_amount,

    CASE 
        WHEN TRIM(total_amount) IS NULL OR TRIM(total_amount) = '' 
            THEN CAST(999.999 AS DECIMAL(10,2))
        ELSE CAST(TRIM(total_amount) AS DECIMAL(10,2))
    END AS total_amount,

    CASE 
        WHEN is_refunded IS NULL THEN 0
        ELSE is_refunded
    END AS is_refunded,

    CASE 
        WHEN TRIM(refund_date) IS NULL OR TRIM(refund_date) = '' 
            THEN CAST('1900-01-01' AS TIMESTAMP)
        ELSE TO_TIMESTAMP(TRIM(refund_date))
    END AS refund_date,

    CASE 
        WHEN TRIM(payment_method) IS NULL OR TRIM(payment_method) IN ('', 'N.A') 
            THEN 'N.A'
        ELSE LOWER(TRIM(payment_method))
    END AS payment_method ,
    ModifiedDate
FROM vw_orders_raw

""")

StatementMeta(, 2b2b6f3d-da55-457b-a1f7-ef908c1628f0, 7, Finished, Available, Finished, False)

In [6]:
DeltaTable.forName(spark, "GamingPlatform_STG.dbo.orders") \
    .alias("t") \
    .merge(
        df_Orders_Cleaned.alias("s"),
        "t.order_id = s.order_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

StatementMeta(, 2b2b6f3d-da55-457b-a1f7-ef908c1628f0, 8, Finished, Available, Finished, False)

## Cities Data CLeansing

In [7]:
# load Data in dataframe
df_cities = spark.sql(f"""
SELECT *
FROM GamingPlatform_ODS.dbo.cities
WHERE ModifiedDate > '{last_load_date}'
""")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 9, Finished, Available, Finished, False)

In [8]:
#create View from DataFrame
df_cities.createOrReplaceTempView("vw_cities_raw")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 10, Finished, Available, Finished, False)

In [9]:
# 3. Clean Data
df_cities_Cleaned = spark.sql("""
SELECT DISTINCT

    CASE 
        WHEN TRIM(city_id) IS NULL OR TRIM(city_id) = '' 
            THEN 999999
        ELSE CAST(TRIM(city_id) AS INT)
    END AS city_id,

    CASE 
        WHEN TRIM(city_name) IS NULL OR TRIM(city_name) IN ('', 'N.A') 
            THEN 'N.A'
        ELSE LOWER(TRIM(city_name))
    END AS city_name,

    CASE 
        WHEN TRIM(state_id) IS NULL OR TRIM(state_id) = '' 
            THEN 999999
        ELSE CAST(TRIM(state_id) AS INT)
    END AS state_id,

    ModifiedDate

FROM vw_cities_raw
ORDER BY city_id
""")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 11, Finished, Available, Finished, False)

In [10]:
DeltaTable.forName(spark, "GamingPlatform_STG.dbo.cities") \
    .alias("t") \
    .merge(
        df_cities_Cleaned.alias("s"),
        "t.city_id = s.city_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 12, Finished, Available, Finished, False)

## Countries Data CLeansing

In [11]:
# load Data in dataframe
df_countries = spark.sql(f"""
SELECT *
FROM GamingPlatform_ODS.dbo.countries
WHERE ModifiedDate > '{last_load_date}'
""")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 13, Finished, Available, Finished, False)

In [12]:
#create View from DataFrame
df_countries.createOrReplaceTempView("vw_countries_raw")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 14, Finished, Available, Finished, False)

In [13]:
# 3. Clean Data
df_countries_Cleaned = spark.sql("""
SELECT DISTINCT

    CASE 
        WHEN TRIM(country_id) IS NULL OR TRIM(country_id) = '' 
            THEN 999999
        ELSE CAST(TRIM(country_id) AS INT)
    END AS country_id,

    CASE 
        WHEN TRIM(country_name) IS NULL OR TRIM(country_name) IN ('', 'N.A') 
            THEN 'N.A'
        ELSE LOWER(TRIM(country_name))
    END AS country_name,

    CASE 
        WHEN TRIM(region) IS NULL OR TRIM(region) IN ('', 'N.A') 
            THEN 'N.A'
        ELSE LOWER(TRIM(region))
    END AS region,

    ModifiedDate

FROM vw_countries_raw
ORDER BY country_id
""")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 15, Finished, Available, Finished, False)

In [14]:
DeltaTable.forName(spark, "GamingPlatform_STG.dbo.countries") \
    .alias("t") \
    .merge(
        df_countries_Cleaned.alias("s"),
        "t.country_id = s.country_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 16, Finished, Available, Finished, False)

## game_genres Data CLeansing

In [15]:
# load Data in dataframe
df_game_genres = spark.sql(f"""
SELECT *
FROM GamingPlatform_ODS.dbo.game_genres
WHERE ModifiedDate > '{last_load_date}'
""")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 17, Finished, Available, Finished, False)

In [16]:
#create View from DataFrame
df_game_genres.createOrReplaceTempView("vw_game_genres_raw")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 18, Finished, Available, Finished, False)

In [17]:
# 3. Clean Data
df_game_genres_Cleaned = spark.sql("""
   SELECT DISTINCT

    CASE 
        WHEN TRIM(game_id) IS NULL OR TRIM(game_id) = '' 
            THEN 999999
        ELSE CAST(TRIM(game_id) AS INT)
    END AS game_id,

    CASE 
        WHEN TRIM(genre) IS NULL OR TRIM(genre) IN ('', 'N.A') 
            THEN 'N.A'
        ELSE LOWER(REPLACE(TRIM(genre), '.', ''))
    END AS genre,

    ModifiedDate

FROM vw_game_genres_raw
ORDER BY game_id
""")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 19, Finished, Available, Finished, False)

In [18]:
DeltaTable.forName(spark, "GamingPlatform_STG.dbo.game_genres") \
    .alias("t") \
    .merge(
        df_game_genres_Cleaned.alias("s"),
        "t.game_id = s.game_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 20, Finished, Available, Finished, False)

## game_metadata Data CLeansing

In [19]:
# load Data in dataframe
df_game_metadata = spark.sql(f"""
SELECT *
FROM GamingPlatform_ODS.dbo.game_metadata
WHERE ModifiedDate > '{last_load_date}'
""")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 21, Finished, Available, Finished, False)

In [20]:
#create View from DataFrame
df_game_metadata.createOrReplaceTempView("vw_game_metadata_raw")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 22, Finished, Available, Finished, False)

In [21]:
# 3. Clean Data
df_game_metadata_Cleaned = spark.sql("""
   SELECT DISTINCT

    CASE 
        WHEN TRIM(game_id) IS NULL OR TRIM(game_id) = '' 
            THEN 999999
        ELSE CAST(TRIM(game_id) AS INT)
    END AS game_id,

    CASE 
        WHEN TRIM(release_date) IS NULL OR TRIM(release_date) = '' 
            THEN CAST('1900-01-01' AS DATE)
        ELSE CAST(TRIM(release_date) AS DATE)
    END AS release_date,

    CASE 
        WHEN TRIM(platform) IS NULL OR TRIM(platform) IN ('', 'N.A') 
            THEN 'N.A'
        ELSE LOWER(REPLACE(TRIM(platform), '.', ''))
    END AS platform,

    CASE 
        WHEN TRIM(rating) IS NULL OR TRIM(rating) = '' 
            THEN 999.9
        ELSE CAST(TRIM(rating) AS DECIMAL(3,1))
    END AS rating,

    ModifiedDate

FROM vw_game_metadata_raw
ORDER BY game_id
""")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 23, Finished, Available, Finished, False)

In [22]:
DeltaTable.forName(spark, "GamingPlatform_STG.dbo.game_metadata") \
    .alias("t") \
    .merge(
        df_game_metadata_Cleaned.alias("s"),
        "t.game_id = s.game_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 24, Finished, Available, Finished, False)

## game_prices Data CLeansing

In [23]:
# load Data in dataframe
df_game_prices = spark.sql(f"""
SELECT *
FROM GamingPlatform_ODS.dbo.game_prices
WHERE ModifiedDate > '{last_load_date}'
""")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 25, Finished, Available, Finished, False)

In [24]:
#create View from DataFrame
df_game_prices.createOrReplaceTempView("vw_game_prices_raw")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 26, Finished, Available, Finished, False)

In [25]:
# 3. Clean Data
df_game_prices_Cleaned = spark.sql("""
   SELECT DISTINCT

    CASE 
        WHEN TRIM(game_id) IS NULL OR TRIM(game_id) = '' 
            THEN 999999
        ELSE CAST(TRIM(game_id) AS INT)
    END AS game_id,

    CASE 
        WHEN TRIM(price) IS NULL OR TRIM(price) = '' 
            THEN 999.999
        ELSE CAST(TRIM(price) AS DECIMAL(10,2))
    END AS price,

    CASE 
        WHEN TRIM(discount_percentage) IS NULL OR TRIM(discount_percentage) = '' 
            THEN 999.999
        ELSE CAST(TRIM(discount_percentage) AS DECIMAL(5,2))
    END AS discount_percentage,

    CASE 
        WHEN TRIM(tax_percentage) IS NULL OR TRIM(tax_percentage) = '' 
            THEN 999.999
        ELSE CAST(TRIM(tax_percentage) AS DECIMAL(5,2))
    END AS tax_percentage,

    ModifiedDate

FROM vw_game_prices_raw
ORDER BY game_id
""")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 27, Finished, Available, Finished, False)

In [26]:
DeltaTable.forName(spark, "GamingPlatform_STG.dbo.game_prices") \
    .alias("t") \
    .merge(
        df_game_prices_Cleaned.alias("s"),
        "t.game_id = s.game_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 28, Finished, Available, Finished, False)

## game_sessions

In [27]:
# load Data in dataframe
df_game_sessions = spark.sql(f"""
SELECT *
FROM GamingPlatform_ODS.dbo.game_sessions
WHERE ModifiedDate > '{last_load_date}'
""")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 29, Finished, Available, Finished, False)

In [28]:
#create View from DataFrame
df_game_sessions.createOrReplaceTempView("vw_game_sessions_raw")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 30, Finished, Available, Finished, False)

In [29]:
# 3. Clean Data
df_game_sessions_Cleaned = spark.sql("""
   SELECT DISTINCT

    CASE 
        WHEN TRIM(session_id) IS NULL OR TRIM(session_id) = '' 
            THEN 999999
        ELSE CAST(TRIM(session_id) AS INT)
    END AS session_id,

    CASE 
        WHEN TRIM(user_id) IS NULL OR TRIM(user_id) = '' 
            THEN 999999
        ELSE CAST(TRIM(user_id) AS INT)
    END AS user_id,

    CASE 
        WHEN TRIM(game_id) IS NULL OR TRIM(game_id) = '' 
            THEN 999999
        ELSE CAST(TRIM(game_id) AS INT)
    END AS game_id,

    CASE 
        WHEN TRIM(hours_played) IS NULL OR TRIM(hours_played) = '' 
            THEN 999.999
        ELSE CAST(TRIM(hours_played) AS DECIMAL(10,2))
    END AS hours_played,

    CASE 
        WHEN session_date IS NULL OR TRIM(session_date) = '' 
            THEN CAST('1900-01-01' AS DATE)
        ELSE CAST(session_date AS DATE)
    END AS session_date,

    CASE 
        WHEN TRIM(device_type) IS NULL OR TRIM(device_type) IN ('', 'N.A') 
            THEN 'N.A'
        ELSE LOWER(REPLACE(TRIM(device_type), '.', ''))
    END AS device_type,

    ModifiedDate

FROM vw_game_sessions_raw
""")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 31, Finished, Available, Finished, False)

In [30]:
DeltaTable.forName(spark, "GamingPlatform_STG.dbo.game_sessions") \
    .alias("t") \
    .merge(
        df_game_sessions_Cleaned.alias("s"),
        "t.session_id = s.session_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 32, Finished, Available, Finished, False)

## game_titles

In [31]:
# load Data in dataframe
df_game_titles = spark.sql(f"""
SELECT *
FROM GamingPlatform_ODS.dbo.game_titles
WHERE ModifiedDate > '{last_load_date}'
""")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 33, Finished, Available, Finished, False)

In [32]:
#create View from DataFrame
df_game_titles.createOrReplaceTempView("vw_game_titles_raw")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 34, Finished, Available, Finished, False)

In [33]:

# 3. Clean Data
df_game_titles_Cleaned = spark.sql("""
   SELECT DISTINCT 

    CASE 
        WHEN TRIM(game_id) IS NULL OR TRIM(game_id) = '' THEN 999999
        ELSE CAST(TRIM(game_id) AS INT)
    END AS game_id,

    CASE 
        WHEN TRIM(game_name) IS NULL OR TRIM(game_name) = '' THEN 'N.A'
        ELSE LOWER(REPLACE(TRIM(game_name), '.', ''))
    END AS game_name,

    ModifiedDate

FROM vw_game_titles_raw
""")


StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 35, Finished, Available, Finished, False)

In [34]:
DeltaTable.forName(spark, "GamingPlatform_STG.dbo.game_titles") \
    .alias("t") \
    .merge(
        df_game_titles_Cleaned.alias("s"),
        "t.game_id = s.game_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 36, Finished, Available, Finished, False)

## states

In [35]:
# load Data in dataframe
df_states = spark.sql(f"""
SELECT *
FROM GamingPlatform_ODS.dbo.states
WHERE ModifiedDate > '{last_load_date}'
""")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 37, Finished, Available, Finished, False)

In [36]:
#create View from DataFrame
df_states.createOrReplaceTempView("vw_states_raw")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 38, Finished, Available, Finished, False)

In [37]:
# 3. Clean Data
df_states_Cleaned = spark.sql("""
   SELECT DISTINCT

    CASE 
        WHEN TRIM(state_id) IS NULL OR TRIM(state_id) = '' 
            THEN 999999
        ELSE CAST(TRIM(state_id) AS INT)
    END AS state_id,

    CASE 
        WHEN TRIM(state_name) IS NULL OR TRIM(state_name) IN ('', 'N.A') 
            THEN 'N.A'
        ELSE LOWER(TRIM(state_name))
    END AS state_name,

    CASE 
        WHEN TRIM(country_id) IS NULL OR TRIM(country_id) = '' 
            THEN 999999
        ELSE CAST(TRIM(country_id) AS INT)
    END AS country_id,

    ModifiedDate

FROM vw_states_raw
ORDER BY state_id
""")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 39, Finished, Available, Finished, False)

In [38]:
DeltaTable.forName(spark, "GamingPlatform_STG.dbo.states") \
    .alias("t") \
    .merge(
        df_states_Cleaned.alias("s"),
        "t.state_id = s.state_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 40, Finished, Available, Finished, False)

## user_activity

In [39]:
# load Data in dataframe
df_user_activity = spark.sql(f"""
SELECT *
FROM GamingPlatform_ODS.dbo.user_activity
WHERE ModifiedDate > '{last_load_date}'
""")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 41, Finished, Available, Finished, False)

In [40]:
# create View from DataFrame
df_user_activity.createOrReplaceTempView("vw_user_activity_raw")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 42, Finished, Available, Finished, False)

In [41]:
# 3. Clean Data
df_user_activity_Cleaned = spark.sql("""
   SELECT DISTINCT

    CASE 
        WHEN TRIM(user_id) IS NULL OR TRIM(user_id) = '' 
            THEN 999999
        ELSE CAST(TRIM(user_id) AS INT)
    END AS user_id,

    CASE 
        WHEN TRIM(created_at) IS NULL OR TRIM(created_at) = '' 
            THEN CAST('1900-01-01' AS TIMESTAMP)
        ELSE TO_TIMESTAMP(TRIM(created_at))
    END AS created_at,

    ModifiedDate

FROM vw_user_activity_raw
ORDER BY user_id
""")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 43, Finished, Available, Finished, False)

In [42]:
DeltaTable.forName(spark, "GamingPlatform_STG.dbo.user_activity") \
    .alias("t") \
    .merge(
        df_user_activity_Cleaned.alias("s"),
        "t.user_id = s.user_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 44, Finished, Available, Finished, False)

## user_basic

In [43]:
# load Data in dataframe
df_user_basic = spark.sql(f"""
SELECT *
FROM GamingPlatform_ODS.dbo.user_basic
WHERE ModifiedDate > '{last_load_date}'
""")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 45, Finished, Available, Finished, False)

In [44]:
# create View from DataFrame
df_user_basic.createOrReplaceTempView("vw_user_basic_raw")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 46, Finished, Available, Finished, False)

In [45]:
# 3. Clean Data
df_user_basic_Cleaned = spark.sql("""
   SELECT DISTINCT

    CASE 
        WHEN TRIM(user_id) IS NULL OR TRIM(user_id) = '' 
            THEN 999999
        ELSE CAST(TRIM(user_id) AS INT)
    END AS user_id,

    CASE 
        WHEN TRIM(username) IS NULL OR TRIM(username) IN ('', 'N.A') 
            THEN 'N.A'
        ELSE LOWER(REPLACE(TRIM(username), '.', ''))
    END AS username,

    CASE 
        WHEN TRIM(status) IS NULL OR TRIM(status) IN ('', 'N.A') 
            THEN 'N.A'
        ELSE LOWER(REPLACE(TRIM(status), '.', ''))
    END AS status,

    ModifiedDate

FROM vw_user_basic_raw
ORDER BY user_id
""")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 47, Finished, Available, Finished, False)

In [46]:
DeltaTable.forName(spark, "GamingPlatform_STG.dbo.user_basic") \
    .alias("t") \
    .merge(
        df_user_basic_Cleaned.alias("s"),
        "t.user_id = s.user_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 48, Finished, Available, Finished, False)

## user_contact

In [47]:
# load Data in dataframe
df_user_contact = spark.sql(f"""
SELECT *
FROM GamingPlatform_ODS.dbo.user_contact
WHERE ModifiedDate > '{last_load_date}'
""")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 49, Finished, Available, Finished, False)

In [48]:
# create View from DataFrame
df_user_contact.createOrReplaceTempView("vw_user_contact_raw")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 50, Finished, Available, Finished, False)

In [49]:
# 3. Clean Data
df_user_contact_Cleaned = spark.sql("""
   SELECT DISTINCT

    CASE 
        WHEN TRIM(user_id) IS NULL OR TRIM(user_id) = '' 
            THEN 999999
        ELSE CAST(TRIM(user_id) AS INT)
    END AS user_id,

    CASE 
        WHEN TRIM(email) IS NULL OR TRIM(email) IN ('', 'N.A') 
            THEN 'N.A'
        ELSE LOWER(TRIM(email))
    END AS email,

    CASE 
        WHEN TRIM(phone) IS NULL OR TRIM(phone) IN ('', 'N.A') 
            THEN 'N.A'
        ELSE TRIM(phone)
    END AS phone,

    ModifiedDate

FROM vw_user_contact_raw
ORDER BY user_id
""")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 51, Finished, Available, Finished, False)

In [50]:
DeltaTable.forName(spark, "GamingPlatform_STG.dbo.user_contact") \
    .alias("t") \
    .merge(
        df_user_contact_Cleaned.alias("s"),
        "t.user_id = s.user_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 52, Finished, Available, Finished, False)

## user_location

In [51]:
# load Data in dataframe
df_user_location = spark.sql(f"""
SELECT *
FROM GamingPlatform_ODS.dbo.user_location
WHERE ModifiedDate > '{last_load_date}'
""")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 53, Finished, Available, Finished, False)

In [52]:
# create View from DataFrame
df_user_location.createOrReplaceTempView("vw_user_location_raw")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 54, Finished, Available, Finished, False)

In [53]:

# 3. Clean Data
df_user_location_Cleaned = spark.sql("""
   SELECT DISTINCT

    CASE 
        WHEN TRIM(user_id) IS NULL OR TRIM(user_id) = '' 
            THEN 999999
        ELSE CAST(TRIM(user_id) AS INT)
    END AS user_id,

    CASE 
        WHEN TRIM(city_id) IS NULL OR TRIM(city_id) = '' 
            THEN 999999
        ELSE CAST(TRIM(city_id) AS INT)
    END AS city_id,

    ModifiedDate

FROM vw_user_location_raw
ORDER BY user_id
""")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 55, Finished, Available, Finished, False)

In [54]:
DeltaTable.forName(spark, "GamingPlatform_STG.dbo.user_location") \
    .alias("t") \
    .merge(
        df_user_location_Cleaned.alias("s"),
        "t.user_id = s.user_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 56, Finished, Available, Finished, False)

## trophies

In [3]:
# load Data in dataframe
df_trophies = spark.sql(f"""
SELECT *
FROM GamingPlatform_ODS.dbo.trophies
WHERE ModifiedDate > '{last_load_date}'
""")

StatementMeta(, dc54fb12-0c47-4ccb-b574-27473e45d332, 5, Finished, Available, Finished, False)

In [4]:
# create View from DataFrame
df_trophies.createOrReplaceTempView("vw_trophies_raw")

StatementMeta(, dc54fb12-0c47-4ccb-b574-27473e45d332, 6, Finished, Available, Finished, False)

In [5]:
# 3. Clean Data
df_trophies_Cleaned = spark.sql("""
WITH CleanedTrophies AS (

    SELECT DISTINCT

        CASE 
            WHEN TRIM(user_id) IS NULL OR TRIM(user_id) = '' 
                THEN 999999
            ELSE CAST(TRIM(user_id) AS INT)
        END AS user_id,

        CASE 
            WHEN TRIM(game_id) IS NULL OR TRIM(game_id) = '' 
                THEN 999999
            ELSE CAST(TRIM(game_id) AS INT)
        END AS game_id,

        NULLIF(
            CASE 
                WHEN TRIM(trophy_name) IS NULL 
                     OR TRIM(trophy_name) IN ('', 'N.A') 
                    THEN 'N.A'
                ELSE LOWER(REPLACE(TRIM(trophy_name), '.', ''))
            END,
        'N.A') AS trophy_name,

        NULLIF(
            CASE 
                WHEN TRIM(trophy_type) IS NULL 
                     OR TRIM(trophy_type) IN ('', 'N.A') 
                    THEN 'N.A'
                ELSE LOWER(REPLACE(TRIM(trophy_type), '.', ''))
            END,
        'N.A') AS trophy_type,

        CASE 
            WHEN TRIM(earned_date) IS NULL OR TRIM(earned_date) = '' 
                THEN CAST('1900-01-01' AS TIMESTAMP)
            ELSE TO_TIMESTAMP(TRIM(earned_date))
        END AS earned_date,

        ModifiedDate

    FROM vw_trophies_raw
),

NumberedTrophies AS (

    SELECT
        CAST(
            ROW_NUMBER() OVER (
                ORDER BY trophy_name, trophy_type
            ) AS INT
        ) AS trophy_id,

        user_id,
        game_id,
        trophy_name,
        trophy_type,
        earned_date,
        ModifiedDate

    FROM CleanedTrophies
)

SELECT *
FROM NumberedTrophies
ORDER BY trophy_id
""")

StatementMeta(, dc54fb12-0c47-4ccb-b574-27473e45d332, 7, Finished, Available, Finished, False)

In [6]:
DeltaTable.forName(spark, "GamingPlatform_STG.dbo.trophies") \
    .alias("t") \
    .merge(
        df_trophies_Cleaned.alias("s"),
        """
        t.user_id = s.user_id
        AND t.game_id = s.game_id
        AND t.trophy_name = s.trophy_name
        AND t.trophy_type = s.trophy_type
        AND t.earned_date = s.earned_date
        """
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

StatementMeta(, dc54fb12-0c47-4ccb-b574-27473e45d332, 8, Finished, Available, Finished, False)

## update Last_Load_Date

In [59]:
from datetime import datetime, timezone

current_time = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

spark.sql(f"""
    UPDATE GamingPlatform_STG.dbo.ETL_Metadata
    SET Last_Load_Date = '{current_time}'
    WHERE Stage = 'STG_Gaming'
""")

print(f"STG_Gaming updated  {current_time}")

StatementMeta(, 7ca39391-2fc6-4e51-bda3-c27b2d3dfc64, 61, Finished, Available, Finished, False)

STG_Gaming updated  2026-05-17 12:40:16


#

## Stop Spark Session

In [1]:
# mssparkutils.session.stop()

StatementMeta(, 77748343-a463-49f7-81ad-7f45de4a2d01, 3, Finished, Available, Finished, False)